In [1]:
import pandas as pd
from pathlib import Path

# Paths to deduped parquet files
base_dir = Path("../data/Postings")

files = [
    base_dir / "SG-2022-WITHOUT-REPOSTS-W60D.parquet",
    base_dir / "SG-2023-WITHOUT-REPOSTS-W60D.parquet",
    base_dir / "SG-2024-WITHOUT-REPOSTS-W60D.parquet",
]

# Read and concatenate
data_df = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print("Combined data_df shape:", data_df.shape)
print("Years covered:", sorted(data_df["post_date"].dt.year.unique()))

Combined data_df shape: (1492675, 41)
Years covered: [2022, 2023, 2024]


In [2]:
recruitment_industries = [
    'Recruitment and Staffing Services',
    'Employment and Staffing Services',
    'Employment and Recruitment Services',
    'Human Resources and Recruitment Services',
    'Online Employment Platforms',
    'Human Resources and Workforce Solutions',
    'Business Process Outsourcing Services'
]

In [3]:
d_unique_filtered = data_df[~data_df['rics_k400'].isin(recruitment_industries)].copy()

In [4]:
import numpy as np

def build_occ_ind_quarter_panel_whitecollar_treat(
    d_unique: pd.DataFrame,
    occ_level_path: str,
    date_col: str = "post_date",
    onet_postings_col: str = "onet_code",
    onet_occ_col: str = "O*NET-SOC Code",
    ind_col: str = "rics_k50",
    firm_col: str = "ultimate_parent_company_name",
    exclude_firms: list[str] | None = None,
    treat_date: str = "2022-11-30",
    out_path: str = "occ_ind_quarter_panel_whitecollar_treat_no_recruit.dta",
    min_qtr: str = "2022Q1",
    max_qtr: str | None = None,
):
    d = d_unique.copy()

    # --------------------------------------------------
    # 0) Optional firm exclusion (unchanged behavior)
    # --------------------------------------------------
    if exclude_firms:
        if firm_col not in d.columns:
            print(f"Warning: firm_col '{firm_col}' not found; firm exclusion skipped.")
        else:
            before = len(d)
            d_firm = d[firm_col].astype(str).str.strip().str.casefold()
            exclude_set = {str(x).strip().casefold() for x in exclude_firms}
            mask_excl = d_firm.isin(exclude_set)
            d = d.loc[~mask_excl].copy()
            removed = int(mask_excl.sum())
            print(f"Excluded firms: {removed} rows ({removed / before:.2%})")

    # --------------------------------------------------
    # 1) Dates
    # --------------------------------------------------
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d = d.dropna(subset=[date_col]).copy()
    treat_dt = pd.Timestamp(treat_date)

    # --------------------------------------------------
    # 2) Load occupation-level white-collar treatment table
    # --------------------------------------------------
    occ = pd.read_csv(occ_level_path)

    if "white_collar_treat" not in occ.columns:
        raise ValueError("white_collar_treat not found in occupation-level file.")

    d[onet_postings_col] = d[onet_postings_col].astype(str).str.strip()
    occ[onet_occ_col] = occ[onet_occ_col].astype(str).str.strip()

    occ_sub = occ[[onet_occ_col, "white_collar_treat"]].rename(
        columns={onet_occ_col: onet_postings_col}
    )

    d = d.merge(occ_sub, on=onet_postings_col, how="left", indicator=True)
    d = d[d["_merge"] == "both"].drop(columns=["_merge"]).copy()

    # --------------------------------------------------
    # 3) Quarter variables
    # --------------------------------------------------
    d["qtr"] = d[date_col].dt.to_period("Q")
    d = d[d["qtr"] >= pd.Period(min_qtr, freq="Q")]
    if max_qtr:
        d = d[d["qtr"] <= pd.Period(max_qtr, freq="Q")]

    # --------------------------------------------------
    # 4) Aggregate to occ–ind–quarter
    # --------------------------------------------------
    panel = (
        d.groupby([onet_postings_col, ind_col, "qtr"])
        .agg(
            num_postings=("job_id", "count"),
            treated=("white_collar_treat", "first"),
        )
        .reset_index()
    )

    panel["treated"] = panel["treated"].fillna(0).astype(int)

    # --------------------------------------------------
    # 5) Balance panel
    # --------------------------------------------------
    occ_ind_pairs = panel[[onet_postings_col, ind_col, "treated"]].drop_duplicates()
    all_qtrs = pd.period_range(panel["qtr"].min(), panel["qtr"].max(), freq="Q")

    full = (
        occ_ind_pairs.assign(_tmp=1)
        .merge(pd.DataFrame({"qtr": all_qtrs, "_tmp": 1}), on="_tmp")
        .drop(columns="_tmp")
    )

    panel = full.merge(
        panel,
        on=[onet_postings_col, ind_col, "qtr", "treated"],
        how="left"
    )

    panel["num_postings"] = panel["num_postings"].fillna(0).astype(int)

    # --------------------------------------------------
    # 6) Treatment timing (2022Q4 inclusive)
    # --------------------------------------------------
    TREAT_QTR = pd.Period("2022Q4", freq="Q")
    panel["post"] = (panel["qtr"] >= TREAT_QTR).astype(int)
    panel["did"] = panel["treated"] * panel["post"]
    panel["log_postings"] = np.log(panel["num_postings"] + 1)

    # --------------------------------------------------
    # 7) Stata-aligned time vars
    # --------------------------------------------------
    panel["year"] = panel["qtr"].dt.year
    panel["qtr_num"] = panel["qtr"].dt.quarter
    panel["tq"] = (panel["year"] - 1960) * 4 + (panel["qtr_num"] - 1)

    base_tq = (2022 - 1960) * 4
    event_tq = base_tq + 3

    panel["t_index"] = panel["tq"] - base_tq
    panel["relq"] = panel["tq"] - event_tq

    # --------------------------------------------------
    # 8) IDs for FE
    # --------------------------------------------------
    panel["occ_id"] = panel[onet_postings_col].astype("category").cat.codes + 1
    panel["ind_id"] = panel[ind_col].astype("category").cat.codes + 1
    panel["occ_ind_id"] = (
        panel[onet_postings_col].astype(str) + "|" + panel[ind_col].astype(str)
    ).astype("category").cat.codes + 1

    panel["qtr_str"] = panel["qtr"].astype(str)
    panel = panel.drop(columns="qtr").sort_values(
        ["occ_ind_id", "year", "qtr_num"]
    )

    panel.to_stata(out_path, write_index=False, version=118)

    print(f"Saved: {out_path}")

    return panel


In [ ]:
panel_whitecollar = build_occ_ind_quarter_panel_whitecollar_treat(
    d_unique=d_unique_filtered,
    occ_level_path="../data/occ_white_collar_treatment.csv",
    out_path="../data/occ_ind_quarter_panel_whitecollar_treat_no_recruit.dta",
    min_qtr="2022Q1"
)

Saved: occ_ind_quarter_panel_whitecollar_treat_no_recruit.dta
